In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 391 unique observed indicators.


In [3]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,summary,...,sha256,size,source,lastObserved,url,text,address,Subject,Block,indicator
34,6755399448037614,2025-05-28T19:30:52Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-29T12:33:06Z,3.0,80.0,64.62.197.226,...,NaN,NaN,NaN,2025-05-29T00:00:00Z,NaN,NaN,NaN,NaN,NaN,64.62.197.226
35,6755399448037610,2025-05-28T19:30:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-29T12:24:54Z,3.0,80.0,193.32.162.96,...,NaN,NaN,NaN,2025-05-29T00:00:00Z,NaN,NaN,NaN,NaN,NaN,193.32.162.96
36,6755399448005402,2025-05-19T11:53:21Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-29T12:24:53Z,3.0,80.0,104.244.79.50,...,NaN,NaN,NaN,2025-05-29T00:00:00Z,NaN,NaN,NaN,NaN,NaN,104.244.79.50
37,5629499542036636,2025-05-28T19:30:50Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-29T12:24:34Z,3.0,80.0,193.163.125.39,...,NaN,NaN,NaN,2025-05-29T00:00:00Z,NaN,NaN,NaN,NaN,NaN,193.163.125.39
38,5629499542036634,2025-05-28T19:30:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-29T12:24:34Z,3.0,80.0,193.163.125.31,...,NaN,NaN,NaN,2025-05-29T00:00:00Z,NaN,NaN,NaN,NaN,NaN,193.163.125.31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9190,4553620,2024-03-29T13:13:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T06:14:00Z,3.0,7.0,169.150.196.152,...,NaN,NaN,NaN,2025-05-27T00:00:00Z,NaN,NaN,NaN,NaN,NaN,169.150.196.152
9487,4515497,2024-02-01T17:29:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T06:13:57Z,3.0,30.0,68.71.254.6,...,NaN,NaN,NaN,2025-05-27T00:00:00Z,NaN,NaN,NaN,NaN,NaN,68.71.254.6
9511,6755399444006850,2025-04-23T15:00:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T06:13:56Z,5.0,96.0,43.225.189.163,...,NaN,NaN,NaN,2025-05-28T00:00:00Z,NaN,NaN,NaN,NaN,NaN,43.225.189.163
9663,4697065,2024-06-05T13:07:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T06:13:55Z,5.0,96.0,37.19.210.21,...,NaN,NaN,NaN,2025-05-28T00:00:00Z,NaN,NaN,NaN,NaN,NaN,37.19.210.21


In [4]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=5)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250529.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250528.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250527.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250526.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250525.csv']
Loaded data from 5 files.


In [5]:
import pandas as pd
from datetime import timedelta, date

# Define cutoff time in UTC
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    
    if isinstance(tags_data, list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)

        # Ensure 'name' column is of string type
        tags['name'] = tags['name'].astype(str)

        # Create a new column with a list of all tag names
        all_tags_list = tags['name'].tolist()

        # Filter for "API" tags
        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

        if not api_tags.empty:
            # Add metadata columns and all_tags list as a single value (not as a series)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink']:
                api_tags[col] = row.get(col)
            
            # Assign the all_tags list as a single value for the entire set of API tags
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)

            # Append to the filtered DataFrame
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Ensure 'lastObserved' is datetime
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Ensure necessary columns exist
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean the 'name' column by removing ' Splunk API'
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

# Initialize the observed_date column with NaN
filtered_tags['observed_date'] = None

# Iterate through each row and search for matching indicator and opdiv
for index, row in filtered_tags.iterrows():
    # Extract summary and cleaned_name
    summary = row['summary']
    cleaned_name = row['cleaned_name']
    
    # Search for matching rows in the observed data
    match = observed_data_df[(observed_data_df['indicator'] == summary) & (observed_data_df['OpDiv'] == cleaned_name)]
    
    # If a match is found, extract the obs_date
    if not match.empty:
        # Assign the first matching obs_date
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

# Drop the 'cleaned_name' column as it is no longer needed
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=72)].copy()

# Make `cutoff` a naive datetime to match the naive `observed_date`
cutoff_naive = cutoff.tz_convert(None)

# Apply the filter directly
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=3)].copy()
# Extract partner names and remove ' Splunk API' suffix
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge partner information back into recent_tags
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Apply additional filters
recent_tags = recent_tags[recent_tags['partner_count'] >= 4]
#recent_tags = recent_tags[recent_tags['observations'] < 15000]

# Filter by dateAdded in the last 30 days
#recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

# Drop duplicates based on 'summary'
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Drop unnecessary columns
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], errors='ignore')


# Remove rows where 'all_tags' contains 'I&W'
#recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x)]
#recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'htoc_wl' in x)]


recent_tags


,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
0,35057,2025-05-29T12:24:54Z,64.62.197.226,3238,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:52+00:00,2025-05-29T12:33:06Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, FDA Splunk API, CMS Splunk AP...",2025-05-29,4,"CMS, FDA, HRSA, IHS"
6,35760,2025-05-29T06:33:33Z,104.244.79.50,417,RITM0581774/TASK0877781,Address,2025-05-19 11:53:21+00:00,2025-05-29T12:24:53Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-29,4,"CMS, HRSA, IHS, OS"
10,471298,2025-05-29T12:33:06Z,193.163.125.39,6355,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:50+00:00,2025-05-29T12:24:34Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,5,"CMS, DHA, FDA, HRSA, OS"
15,471298,2025-05-29T12:33:06Z,193.163.125.31,6440,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:48+00:00,2025-05-29T12:24:34Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"
21,471298,2025-05-29T12:33:06Z,193.163.125.149,6436,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:51+00:00,2025-05-29T12:24:33Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,5,"CMS, DHA, FDA, HRSA, OS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831,471298,2025-05-29T12:33:06Z,96.126.110.22,309751,INC9045771,Address,2025-05-07 12:51:46+00:00,2025-05-27T08:35:46Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-27,6,"CMS, DHA, FDA, HRSA, IHS, OS"
839,35760,2025-05-29T06:33:33Z,45.84.107.128,20875,Executive Summary: \n\nThe following TOR node...,Address,2025-04-25 13:02:39+00:00,2025-05-27T08:35:46Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Active Scanning, FailedLogin, OS Splunk API, ...",2025-05-29,4,"CMS, FDA, HRSA, OS"
843,35760,2025-05-29T06:33:33Z,104.156.155.30,14909,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:47+00:00,2025-05-27T08:35:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-29,4,"CMS, FDA, HRSA, OS"
847,35760,2025-05-29T06:33:33Z,104.152.52.239,270,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:45+00:00,2025-05-27T08:35:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-27,4,"CMS, FDA, HRSA, OS"


In [6]:
import pandas as pd
from datetime import timedelta, date

# Define cutoff time in UTC
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    
    if isinstance(tags_data, list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)

        # Ensure 'name' column is of string type
        tags['name'] = tags['name'].astype(str)

        # Create a new column with a list of all tag names
        all_tags_list = tags['name'].tolist()

        # Filter for "API" tags
        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

        if not api_tags.empty:
            # Add metadata columns and all_tags list as a single value (not as a series)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink']:
                api_tags[col] = row.get(col)
            
            # Assign the all_tags list as a single value for the entire set of API tags
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)

            # Append to the filtered DataFrame
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Ensure 'lastObserved' is datetime
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Ensure necessary columns exist
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean the 'name' column by removing ' Splunk API'
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

# Initialize the observed_date column with NaN
filtered_tags['observed_date'] = None

# Iterate through each row and search for matching indicator and opdiv
for index, row in filtered_tags.iterrows():
    # Extract summary and cleaned_name
    summary = row['summary']
    cleaned_name = row['cleaned_name']
    
    # Search for matching rows in the observed data
    match = observed_data_df[(observed_data_df['indicator'] == summary) & (observed_data_df['OpDiv'] == cleaned_name)]
    
    # If a match is found, extract the obs_date
    if not match.empty:
        # Assign the first matching obs_date
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

# Drop the 'cleaned_name' column as it is no longer needed
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()

# Make `cutoff` a naive datetime to match the naive `observed_date`
cutoff_naive = cutoff.tz_convert(None)

# Apply the filter directly
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()
# Extract partner names and remove ' Splunk API' suffix
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge partner information back into recent_tags
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Drop duplicates based on 'summary'
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Drop unnecessary columns
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], errors='ignore')

recent_tags = recent_tags[recent_tags['all_tags'].apply(lambda x: 'IHS Splunk API' in x)]
recent_tags = recent_tags[recent_tags['partners'].str.contains('IHS')]


recent_tags


,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
0,35057,2025-05-29T12:24:54Z,64.62.197.226,3238,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:52+00:00,2025-05-29T12:33:06Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, FDA Splunk API, CMS Splunk AP...",2025-05-29,4,"CMS, FDA, HRSA, IHS"
6,35760,2025-05-29T06:33:33Z,104.244.79.50,417,RITM0581774/TASK0877781,Address,2025-05-19 11:53:21+00:00,2025-05-29T12:24:53Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-29,2,"IHS, OS"
13,471298,2025-05-29T12:33:06Z,193.163.125.31,6440,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:48+00:00,2025-05-29T12:24:34Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"
29,471298,2025-05-29T12:33:06Z,193.163.125.127,6521,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:47+00:00,2025-05-29T12:24:33Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"
45,35760,2025-05-29T06:33:33Z,193.163.125.74,6777,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:52+00:00,2025-05-29T12:24:31Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-05-29,6,"CMS, FDA, HRSA, IHS, NIH, OS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
692,471298,2025-05-29T12:33:06Z,196.251.69.91,1851568,INC9037042,Address,2025-05-05 15:47:52+00:00,2025-05-27T08:35:47Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"
708,35760,2025-05-29T06:33:33Z,96.126.110.22,309751,INC9045771,Address,2025-05-07 12:51:46+00:00,2025-05-27T08:35:46Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-28,5,"CMS, FDA, HRSA, IHS, OS"
724,471298,2025-05-29T12:33:06Z,104.237.131.164,208186,INC9045771,Address,2025-05-07 12:51:47+00:00,2025-05-27T08:35:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,5,"CMS, DHA, FDA, HRSA, IHS"
734,23586,2025-05-29T07:50:58Z,74.208.236.241,3199,IP address is associated with the domain hosti...,Address,2020-04-23 13:48:58+00:00,2025-05-27T08:35:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[VA OIS CSOC CTS Splunk, HTOC_Testing_Indicato...",2025-05-28,2,"CDC, IHS"


In [15]:
indicators = recent_tags['summary']
indicators

8         68.69.186.86
14     193.163.125.107
18     193.163.125.149
22      193.163.125.42
27       64.62.197.145
            ...       
793      196.251.69.91
813      96.126.110.22
825     104.156.155.30
829     104.152.52.239
833    104.237.131.164
Name: summary, Length: 69, dtype: object

In [16]:
import pandas as pd
import urllib.parse

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)


In [17]:
filtered_recent_tags

,id,description,lastUsed,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
0,471298,CMS received volumetrics alerts about a set of...,2025-05-29T06:33:40Z,83.222.191.10,926746,Address,2025-05-23 12:00:30+00:00,2025-05-28T09:18:27Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning, OS Splunk AP...",2025-05-28,5,"CMS, DHA, HRSA, NIH, OS"
1,471298,INC9043801,2025-05-29T06:33:40Z,192.155.84.28,261427,Address,2025-05-06 13:47:53+00:00,2025-05-28T06:24:37Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"
2,35057,INC9045771,2025-05-29T06:02:32Z,45.135.57.192,102224,Address,2025-05-07 12:51:44+00:00,2025-05-28T06:24:36Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,4,"CMS, FDA, HRSA, IHS"
3,471298,INC9043801,2025-05-29T06:33:40Z,157.230.162.215,330243,Address,2025-05-06 13:47:54+00:00,2025-05-28T06:24:36Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"
4,471298,HTOC—Romanian IP Address Reported in May 2025 ...,2025-05-29T06:33:40Z,83.222.190.238,548713,Address,2025-05-23 12:23:32+00:00,2025-05-27T16:21:59Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning, OS Splunk AP...",2025-05-28,4,"CMS, DHA, HRSA, OS"
5,471298,CMS received volumetrics alerts about a set of...,2025-05-29T06:33:40Z,83.222.191.70,11838038,Address,2025-05-23 12:00:30+00:00,2025-05-27T08:35:50Z,2025-05-28 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning, OS Splunk AP...",2025-05-28,6,"CMS, DHA, FDA, HRSA, IHS, OS"
6,471298,CMS received a set of volumetric alerts about ...,2025-05-29T06:33:40Z,83.222.190.46,10529444,Address,2025-05-23 12:23:32+00:00,2025-05-27T08:35:50Z,2025-05-28 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning, OS Splunk AP...",2025-05-28,7,"CMS, DHA, FDA, HRSA, IHS, NIH, OS"
7,471298,INC9037042,2025-05-29T06:33:40Z,196.251.69.91,1832564,Address,2025-05-05 15:47:52+00:00,2025-05-27T08:35:47Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"
8,471298,INC9045771,2025-05-29T06:33:40Z,96.126.110.22,309749,Address,2025-05-07 12:51:46+00:00,2025-05-27T08:35:46Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-27,6,"CMS, DHA, FDA, HRSA, IHS, OS"
9,471298,INC9045771,2025-05-29T06:33:40Z,104.237.131.164,207208,Address,2025-05-07 12:51:47+00:00,2025-05-27T08:35:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS"


In [7]:
import pandas as pd
import requests
import time
import ipaddress
from datetime import datetime
import concurrent.futures

# API Keys
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"

# Headers for API requests
VT_HEADERS = {"x-apikey": VT_API_KEY}
OTX_HEADERS = {"X-OTX-API-KEY": OTX_API_KEY}

# API URLs
VT_URL_IP = "https://www.virustotal.com/api/v3/ip_addresses/{}"
VT_URL_DOMAIN = "https://www.virustotal.com/api/v3/domains/{}"
OTX_URL_IP = "https://otx.alienvault.io/api/v1/indicators/IPv4/{}/general"
OTX_URL_DOMAIN = "https://otx.alienvault.io/api/v1/indicators/domain/{}/general"
OTX_URL_HOSTNAME = "https://otx.alienvault.io/api/v1/indicators/hostname/{}"

def is_ip(value):
    """ Check if the given value is a valid IP address. """
    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False

def determine_query_type(query):
    """ Determine if the query is an IP, domain, or hostname. """
    if is_ip(query):
        return "ip"
    elif "." in query:
        return "hostname"
    else:
        return "domain"

def fetch_virustotal_data(query):
    """Fetch data from VirusTotal API for IP or Domain."""
    query_type = determine_query_type(query)
    url = VT_URL_IP.format(query) if query_type == "ip" else VT_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=VT_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json().get("data", {}).get("attributes", {})

        return {
            "search_term": query,
            "asn": data.get('asn'),
            "as_owner": data.get('as_owner'),
            "country": data.get('country'),
            "network": data.get('network'),
            "last_analysis_stats": data.get('last_analysis_stats'),
            "reputation": data.get('reputation'),
            "total_votes": data.get('total_votes'),
            "open_ports": [s.get("port") for s in data.get("services", []) if "port" in s],
            "link": f"https://www.virustotal.com/gui/ip-address/{query}" if query_type == "ip" else f"https://www.virustotal.com/gui/domain/{query}"
        }

    except Exception as e:
        print(f"VirusTotal Error for {query}: {e}")
        return None

def fetch_otx_data(query):
    """Fetch data from OTX API for IP, Domain, or Hostname."""
    query_type = determine_query_type(query)

    if query_type == "ip":
        url = OTX_URL_IP.format(query)
    elif query_type == "hostname":
        url = OTX_URL_HOSTNAME.format(query)
    else:
        url = OTX_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=OTX_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json()

        return {
            "search_term": query,
            "base_indicator": data.get('base_indicator', {}),
            "reputation": data.get('reputation'),
            "geo": data.get('geo', {}),
            "whois": data.get('whois', {}),
            "open_ports": data.get('open_ports', []),
            "link": f"https://otx.alienvault.com/indicator/{query_type}/{query}"
        }

    except Exception as e:
        print(f"OTX Error for {query}: {e}")
        return None

def unnest_base_indicator(df):
    """ Unnest the 'base_indicator' column and create separate columns for its keys. """
    if 'base_indicator' not in df.columns:
        return df

    base_df = pd.json_normalize(df['base_indicator'])
    base_df.columns = [f"base_{col}" for col in base_df.columns]

    df = df.drop(columns=['base_indicator']).reset_index(drop=True)
    df = pd.concat([df, base_df], axis=1)
    return df

def process_indicator(indicator, observed_by, partner_count):
    """Fetch data for a single indicator."""
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        vt_future = executor.submit(fetch_virustotal_data, indicator)
        otx_future = executor.submit(fetch_otx_data, indicator)

        vt_data = vt_future.result()
        otx_data = otx_future.result()

    if vt_data:
        vt_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    if otx_data:
        otx_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    return vt_data, otx_data

def main(recent_tags):
    """ Main function to process indicators. """
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing.")
        return pd.DataFrame(), pd.DataFrame()

    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Processing {len(search_terms)} unique search terms.")

    vt_results = []
    otx_results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for indicator in search_terms:
            partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
            partner_count = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
            observed_by = partners[0] if len(partners) > 0 else "N/A"
            partner_count = partner_count[0] if len(partner_count) > 0 else "N/A"

            futures.append(executor.submit(process_indicator, indicator, observed_by, partner_count))

        for future in concurrent.futures.as_completed(futures):
            vt_data, otx_data = future.result()
            if vt_data:
                vt_results.append(vt_data)
            if otx_data:
                otx_results.append(otx_data)

    vt_df = pd.DataFrame(vt_results)
    otx_df = pd.DataFrame(otx_results)

    otx_df = unnest_base_indicator(otx_df)

    return vt_df, otx_df

if __name__ == "__main__":
    vt_df, otx_df = main(recent_tags)
    print("Script completed successfully.")


Processing 73 unique search terms.
Script completed successfully.


In [8]:
vt_df

,search_term,asn,as_owner,country,network,last_analysis_stats,reputation,total_votes,open_ports,link,timestamp,observed_by,partner_count
0,5.188.206.114,200391.0,Krez 999 Eood,BG,5.188.206.0/24,"{'malicious': 7, 'suspicious': 4, 'undetected'...",0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/ip-address/5.18...,2025-05-29 13:35:13,"CMS, DHA, FDA, HRSA, IHS, OS",6
1,45.91.250.107,30823.0,aurologic GmbH,US,45.91.250.0/24,"{'malicious': 5, 'suspicious': 1, 'undetected'...",0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/ip-address/45.9...,2025-05-29 13:35:13,"FDA, IHS",2
2,104.244.79.50,53667.0,PONYNET,LU,104.244.72.0/21,"{'malicious': 8, 'suspicious': 3, 'undetected'...",-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/104....,2025-05-29 13:35:13,"IHS, OS",2
3,193.163.125.31,211298.0,Driftnet Ltd,GB,193.163.125.0/24,"{'malicious': 6, 'suspicious': 2, 'undetected'...",-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/193....,2025-05-29 13:35:13,"CMS, DHA, FDA, HRSA, IHS, OS",6
4,193.163.125.127,211298.0,Driftnet Ltd,GB,193.163.125.0/24,"{'malicious': 10, 'suspicious': 3, 'undetected...",-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/193....,2025-05-29 13:35:13,"CMS, DHA, FDA, HRSA, IHS, OS",6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,64.62.197.39,6939.0,HURRICANE,US,64.62.197.0/24,"{'malicious': 7, 'suspicious': 2, 'undetected'...",-4,"{'harmless': 0, 'malicious': 4}",[],https://www.virustotal.com/gui/ip-address/64.6...,2025-05-29 13:35:24,"CMS, DHA, FDA, HRSA, IHS",5
69,104.234.115.167,396982.0,GOOGLE-CLOUD-PLATFORM,CA,104.234.115.0/24,"{'malicious': 8, 'suspicious': 2, 'undetected'...",-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/104....,2025-05-29 13:35:26,"CMS, DHA, HRSA, IHS",4
70,118.193.59.10,135377.0,UCLOUD INFORMATION TECHNOLOGY HK LIMITED,DE,118.193.56.0/21,"{'malicious': 9, 'suspicious': 3, 'undetected'...",-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/118....,2025-05-29 13:35:25,"CMS, IHS, OS",3
71,104.192.3.74,27176.0,DATAWAGON,US,104.192.0.0/22,"{'malicious': 10, 'suspicious': 3, 'undetected...",-5,"{'harmless': 0, 'malicious': 5}",[],https://www.virustotal.com/gui/ip-address/104....,2025-05-29 13:35:27,"HRSA, IHS",2


In [9]:
# Unnest the 'last_analysis_stats' dictionary into separate columns in vt_df
if 'last_analysis_stats' in vt_df.columns:
    stats_df = pd.json_normalize(vt_df['last_analysis_stats'])
    stats_df.columns = [f'last_analysis_{col}' for col in stats_df.columns]
    vt_df = pd.concat([vt_df.drop(columns=['last_analysis_stats']), stats_df], axis=1)

vt_df

,search_term,asn,as_owner,country,network,reputation,total_votes,open_ports,link,timestamp,observed_by,partner_count,last_analysis_malicious,last_analysis_suspicious,last_analysis_undetected,last_analysis_harmless,last_analysis_timeout
0,5.188.206.114,200391.0,Krez 999 Eood,BG,5.188.206.0/24,0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/ip-address/5.18...,2025-05-29 13:35:13,"CMS, DHA, FDA, HRSA, IHS, OS",6,7,4,28,55,0
1,45.91.250.107,30823.0,aurologic GmbH,US,45.91.250.0/24,0,"{'harmless': 0, 'malicious': 0}",[],https://www.virustotal.com/gui/ip-address/45.9...,2025-05-29 13:35:13,"FDA, IHS",2,5,1,29,59,0
2,104.244.79.50,53667.0,PONYNET,LU,104.244.72.0/21,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/104....,2025-05-29 13:35:13,"IHS, OS",2,8,3,29,54,0
3,193.163.125.31,211298.0,Driftnet Ltd,GB,193.163.125.0/24,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/193....,2025-05-29 13:35:13,"CMS, DHA, FDA, HRSA, IHS, OS",6,6,2,30,56,0
4,193.163.125.127,211298.0,Driftnet Ltd,GB,193.163.125.0/24,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/193....,2025-05-29 13:35:13,"CMS, DHA, FDA, HRSA, IHS, OS",6,10,3,29,52,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,64.62.197.39,6939.0,HURRICANE,US,64.62.197.0/24,-4,"{'harmless': 0, 'malicious': 4}",[],https://www.virustotal.com/gui/ip-address/64.6...,2025-05-29 13:35:24,"CMS, DHA, FDA, HRSA, IHS",5,7,2,29,56,0
69,104.234.115.167,396982.0,GOOGLE-CLOUD-PLATFORM,CA,104.234.115.0/24,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/104....,2025-05-29 13:35:26,"CMS, DHA, HRSA, IHS",4,8,2,30,54,0
70,118.193.59.10,135377.0,UCLOUD INFORMATION TECHNOLOGY HK LIMITED,DE,118.193.56.0/21,-1,"{'harmless': 0, 'malicious': 1}",[],https://www.virustotal.com/gui/ip-address/118....,2025-05-29 13:35:25,"CMS, IHS, OS",3,9,3,29,53,0
71,104.192.3.74,27176.0,DATAWAGON,US,104.192.0.0/22,-5,"{'harmless': 0, 'malicious': 5}",[],https://www.virustotal.com/gui/ip-address/104....,2025-05-29 13:35:27,"HRSA, IHS",2,10,3,28,53,0


In [10]:
# Merge 'last_analysis_malicious' from vt_df into recent_tags based on summary/search_term
recent_tags = recent_tags.merge(
    vt_df[['search_term', 'last_analysis_malicious']],
    left_on='summary',
    right_on='search_term',
    how='left'
).drop(columns=['search_term'])

recent_tags

,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,last_analysis_malicious
0,35057,2025-05-29T12:24:54Z,64.62.197.226,3238,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:52+00:00,2025-05-29T12:33:06Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, FDA Splunk API, CMS Splunk AP...",2025-05-29,4,"CMS, FDA, HRSA, IHS",7
1,35760,2025-05-29T06:33:33Z,104.244.79.50,417,RITM0581774/TASK0877781,Address,2025-05-19 11:53:21+00:00,2025-05-29T12:24:53Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-29,2,"IHS, OS",8
2,471298,2025-05-29T12:33:06Z,193.163.125.31,6440,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:48+00:00,2025-05-29T12:24:34Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS",6
3,471298,2025-05-29T12:33:06Z,193.163.125.127,6521,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:47+00:00,2025-05-29T12:24:33Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS",10
4,35760,2025-05-29T06:33:33Z,193.163.125.74,6777,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:52+00:00,2025-05-29T12:24:31Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-05-29,6,"CMS, FDA, HRSA, IHS, NIH, OS",6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68,471298,2025-05-29T12:33:06Z,196.251.69.91,1851568,INC9037042,Address,2025-05-05 15:47:52+00:00,2025-05-27T08:35:47Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS",10
69,35760,2025-05-29T06:33:33Z,96.126.110.22,309751,INC9045771,Address,2025-05-07 12:51:46+00:00,2025-05-27T08:35:46Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-28,5,"CMS, FDA, HRSA, IHS, OS",9
70,471298,2025-05-29T12:33:06Z,104.237.131.164,208186,INC9045771,Address,2025-05-07 12:51:47+00:00,2025-05-27T08:35:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,5,"CMS, DHA, FDA, HRSA, IHS",9
71,23586,2025-05-29T07:50:58Z,74.208.236.241,3199,IP address is associated with the domain hosti...,Address,2020-04-23 13:48:58+00:00,2025-05-27T08:35:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[VA OIS CSOC CTS Splunk, HTOC_Testing_Indicato...",2025-05-28,2,"CDC, IHS",0


In [11]:
malicious = recent_tags[recent_tags['last_analysis_malicious'] > 10]

malicious

,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,last_analysis_malicious
5,35057,2025-05-29T12:24:54Z,68.69.186.86,7889,INC9072336,Address,2025-05-27 13:19:03+00:00,2025-05-29T12:24:30Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[FDA Splunk API, CMS Splunk API, VA CSOC CTS S...",2025-05-29,5,"CMS, FDA, HRSA, IHS, NIH",11
9,30479,2025-05-29T09:06:19Z,185.129.61.129,236,TASK0881388 / RITM0584642,Address,2025-05-28 17:45:50+00:00,2025-05-29T07:06:45Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[CMS Splunk API, IHS Splunk API, Observed, mal...",2025-05-29,2,"CMS, IHS",11
11,471298,2025-05-29T12:33:06Z,64.62.197.145,22778,RITM0584685 / TASK0881439,Address,2025-05-28 19:30:54+00:00,2025-05-29T06:33:33Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-05-29,6,"CMS, DHA, FDA, HRSA, IHS, OS",11
18,35760,2025-05-29T06:33:33Z,78.153.140.179,9558,RITM0581780/TASK0877793,Address,2025-05-19 11:39:42+00:00,2025-05-28T06:24:42Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-05-29,5,"CMS, FDA, HRSA, IHS, OS",13
21,471298,2025-05-29T12:33:06Z,196.251.83.136,13520895,INC9056503,Address,2025-05-14 13:39:28+00:00,2025-05-28T06:24:41Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,7,"CMS, DHA, FDA, HRSA, IHS, NIH, OS",11
36,35057,2025-05-29T12:24:54Z,45.135.57.192,102619,INC9045771,Address,2025-05-07 12:51:44+00:00,2025-05-28T06:24:36Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,4,"CMS, FDA, HRSA, IHS",13
43,35760,2025-05-29T06:33:33Z,80.67.167.81,13408,Executive Summary: \n\n80.67.167[.]81 resolve...,Address,2025-04-28 19:45:39+00:00,2025-05-28T06:24:35Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[UAC-0082, Malware: ZeroWipe, Malware: CaddyWi...",2025-05-29,5,"CMS, FDA, HRSA, IHS, OS",15
50,471298,2025-05-29T12:33:06Z,142.93.13.29,8948,INC9072336,Address,2025-05-27 13:19:04+00:00,2025-05-28T04:33:16Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, FDA Splunk API, CMS Splunk AP...",2025-05-28,5,"CMS, DHA, FDA, HRSA, IHS",15
51,471298,2025-05-29T12:33:06Z,141.98.11.147,596173,RITM0581780/TASK0877793,Address,2025-05-19 11:57:17+00:00,2025-05-27T08:35:55Z,2025-05-29 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-29,6,"CDC, CMS, DHA, HRSA, IHS, NIH",11
52,471298,2025-05-29T12:33:06Z,141.98.11.82,204773,RITM0582833,Address,2025-05-21 13:34:38+00:00,2025-05-27T08:35:54Z,2025-05-28 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, VA OIS CSOC CTS Splunk, FDA S...",2025-05-28,5,"CMS, DHA, HRSA, IHS, NIH",15


In [20]:
otx_df

,search_term,reputation,geo,whois,open_ports,link,timestamp,observed_by,partner_count,base_id,base_indicator,base_type,base_title,base_description,base_content,base_access_type,base_access_reason
0,193.163.125.39,0,{},http://whois.domaintools.com/193.163.125.39,[],https://otx.alienvault.com/indicator/ip/193.16...,2025-05-29 11:56:49,"CMS, DHA, HRSA, OS",4,3.020768e+09,193.163.125.39,IPv4,,,,public,
1,193.163.125.153,0,{},http://whois.domaintools.com/193.163.125.153,[],https://otx.alienvault.com/indicator/ip/193.16...,2025-05-29 11:56:49,"CMS, DHA, HRSA, OS",4,3.020767e+09,193.163.125.153,IPv4,,,,public,
2,190.202.131.222,0,{},http://whois.domaintools.com/190.202.131.222,[],https://otx.alienvault.com/indicator/ip/190.20...,2025-05-29 11:56:49,"CMS, FDA, HRSA, NIH, OS",5,3.905358e+09,190.202.131.222,IPv4,,,,public,
3,193.163.125.51,0,{},http://whois.domaintools.com/193.163.125.51,[],https://otx.alienvault.com/indicator/ip/193.16...,2025-05-29 11:56:49,"CMS, HRSA, NIH, OS",4,3.015490e+09,193.163.125.51,IPv4,,,,public,
4,68.69.186.86,0,{},http://whois.domaintools.com/68.69.186.86,[],https://otx.alienvault.com/indicator/ip/68.69....,2025-05-29 11:56:49,"CMS, HRSA, IHS, NIH",4,4.011905e+09,68.69.186.86,IPv4,,,,public,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,64.62.197.39,0,{},http://whois.domaintools.com/64.62.197.39,[],https://otx.alienvault.com/indicator/ip/64.62....,2025-05-29 11:56:57,"CMS, DHA, FDA, HRSA, IHS",5,2.938406e+09,64.62.197.39,IPv4,,,,public,
65,141.98.11.82,0,{},http://whois.domaintools.com/141.98.11.82,[],https://otx.alienvault.com/indicator/ip/141.98...,2025-05-29 11:56:55,"CMS, DHA, FDA, HRSA, IHS, NIH",6,3.763666e+09,141.98.11.82,IPv4,,,,public,
66,103.147.185.248,0,{},http://whois.domaintools.com/103.147.185.248,[],https://otx.alienvault.com/indicator/ip/103.14...,2025-05-29 11:56:59,"CMS, FDA, HRSA, OS",4,4.067452e+09,103.147.185.248,IPv4,,,,public,
67,104.234.115.167,0,{},http://whois.domaintools.com/104.234.115.167,[],https://otx.alienvault.com/indicator/ip/104.23...,2025-05-29 11:56:59,"CMS, DHA, FDA, HRSA, IHS",5,4.036630e+09,104.234.115.167,IPv4,,,,public,


In [26]:
import os
import pandas as pd
import docx
from datetime import datetime
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.oxml.shared import OxmlElement

# File paths
#TEMPLATE_PATH = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\I&W Report Template.docx"
TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
#OUTPUT_DIR = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Generated Reports"
OUTPUT_DIR = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def consolidate_sources(vt_df, otx_df):
    """ Consolidate links from both DataFrames for each search term. """
    # Collect links from VirusTotal
    vt_links = vt_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    vt_links.columns = ['search_term', 'vt_links']

    # Collect links from OTX
    otx_links = otx_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    otx_links.columns = ['search_term', 'otx_links']

    # Merge the two link sets
    consolidated = pd.merge(vt_links, otx_links, on='search_term', how='outer')

    # Combine the links, handling NaN values
    consolidated['sources'] = consolidated[['vt_links', 'otx_links']].fillna('').apply(
        lambda x: ', '.join(filter(None, x)), axis=1
    )

    return consolidated[['search_term', 'sources']]

def extract_date(timestamp):
    """ Extract only the date from the timestamp. """
    if pd.isna(timestamp) or timestamp == 'N/A':
        return 'N/A'
    
    # Handle datetime object or string
    try:
        # Attempt to parse as a datetime object
        if isinstance(timestamp, str):
            timestamp = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        return timestamp.strftime("%Y-%m-%d")
    except Exception:
        return 'N/A'

def populate_table(table, data):
    """ Populate a Word table with the given data. """
    # Iterate over data and populate rows
    for index, row in data.iterrows():
        # Insert a new row before the last row (template row)
        new_row = table.add_row().cells
        new_row[0].text = str(row.get('search_term', 'N/A'))
        new_row[1].text = str(row.get('type', 'N/A'))
        new_row[2].text = extract_date(row.get('observed_date', 'N/A'))
        # For the 'observed_by_otx' column, stack values instead of comma-separating
        # observed_by_otx: stack values (one per line) from OTX and VirusTotal "observed_by" columns if present
        observed_by_otx = ''
        observed_by_list = []

        # Try to get observed_by from OTX
        if 'observed_by_otx' in row and pd.notna(row['observed_by_otx']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by_otx']).split(',') if v.strip()])
        elif 'observed_by' in row and pd.notna(row['observed_by']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by']).split(',') if v.strip()])

        # Remove duplicates and join with newlines
        if observed_by_list:
            observed_by_otx = '\n'.join(sorted(set(observed_by_list)))
        else:
            observed_by_otx = 'N/A'
        new_row[3].text = str(observed_by_otx)
        new_row[4].text = str(row.get('notes', ''))
        
def fill_word_template(template_path, output_path, df):
    """ Fill the template with data and place sources outside the table. """
    if not os.path.exists(template_path):
        print(f"Template not found: {template_path}")
        return
    
    try:
        doc = Document(template_path)

        # Populate the table
        table = None
        for tbl in doc.tables:
            if "Indicators/Identifiers" in tbl.rows[0].cells[0].text:
                table = tbl
                break

        if table:
            populate_table(table, df)

        # Find and replace `{{sources}}` placeholder outside the table
        for para in doc.paragraphs:
            if "{{ipaddress}}" in para.text:
                # Try to get the first search_term as the IP address
                ip_address = str(df['search_term'].iloc[0]) if 'search_term' in df.columns else 'N/A'
                para.text = para.text.replace("{{ipaddress}}", ip_address)
            if "{{asn}}" in para.text:
                # Try to get ASN from vt_df if available, else use 'N/A'
                asn_value = str(df['asn'].iloc[0]) if 'asn' in df.columns and not df['asn'].isna().all() else 'N/A'
                para.text = para.text.replace("{{asn}}", asn_value)
            if "{{whois}}" in para.text:
                # Try to get WHOIS info from otx_df if available, else use 'N/A'
                whois_value = str(df['whois'].iloc[0]) if 'whois' in df.columns and not df['whois'].isna().all() else 'N/A'
                para.text = para.text.replace("{{whois}}", whois_value)
            if "{{partners}}" in para.text:
                # Try to get partners from recent_tags if available, else use 'N/A'
                partners_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    partners_row = recent_tags[recent_tags['summary'] == search_term]
                    if not partners_row.empty and 'partners' in partners_row.columns:
                        partners_value = str(partners_row['partners'].iloc[0])
                if not partners_value:
                    partners_value = 'N/A'
                para.text = para.text.replace("{{partners}}", partners_value)
            if "{{weblink}}" in para.text:
                weblink_value = ''
                # Try to get the first search_term as the indicator
                weblink_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    # Try to find a 'webLink' in df for the indicator (if present)
                    if 'webLink' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['webLink'].iloc[0]):
                            weblink_value = str(match['webLink'].iloc[0])
                    # Fallback: try 'link' column if 'webLink' is not present or empty
                    if not weblink_value and 'link' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['link'].iloc[0]):
                            weblink_value = str(match['link'].iloc[0])
                para.text = para.text.replace("{{weblink}}", "")
                if weblink_value:
                    # Add hyperlink using WordprocessingML (only show the link, no display text)
                    r_id = doc.part.relate_to(
                        weblink_value, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True
                    )
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    new_run = OxmlElement('w:r')
                    rPr = OxmlElement('w:rPr')
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    t = OxmlElement('w:t')
                    t.text = weblink_value
                    new_run.append(t)
                    hyperlink.append(new_run)
                    para._p.append(hyperlink)
                else:
                    para.text = "N/A"
            if "{{sources}}" in para.text:
                # Build sources as hyperlinks if possible

                # Helper to add a hyperlink to a paragraph
                def add_hyperlink(paragraph, url):
                    # Create the w:hyperlink tag and add needed values
                    part = paragraph.part
                    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    # Create a w:r element
                    new_run = OxmlElement('w:r')
                    # Create a w:rPr element
                    rPr = OxmlElement('w:rPr')
                    # Add color and underline for hyperlink style
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    # Create a w:t element and set the text
                    t = OxmlElement('w:t')
                    t.text = url
                    new_run.append(t)
                    hyperlink.append(new_run)
                    paragraph._p.append(hyperlink)

                # Remove the placeholder
                para.text = para.text.replace("{{sources}}", "")

                # Add each source as a hyperlink, stacked (one per line, no commas)
                sources = []
                for srcs in df['sources'].dropna().unique():
                    for src in [s.strip() for s in srcs.split(',') if s.strip()]:
                        sources.append(src)
                for i, src in enumerate(sources):
                    add_hyperlink(para, src)
                    if i < len(sources) - 1:
                        para.add_run().add_break()  # Add line break instead of comma

        # --- Save the document ---
        indicator_name = str(df['search_term'].iloc[0]) if 'search_term' in df.columns else 'Unnamed_Indicator'
        sanitized_name = indicator_name.replace(":", "_").replace("/", "_")

        current_date = datetime.now().strftime("%Y-%m-%d")
        folder_path = os.path.join(OUTPUT_DIR, current_date)
        os.makedirs(folder_path, exist_ok=True)

        output_path = os.path.join(folder_path, f"I&W_Report_{sanitized_name}.docx")
        doc.save(output_path)
        print(f"Saved report: {output_path}")

    except Exception as e:
        print(f"Error while generating report for {indicator_name}: {e}")

def main(vt_df, otx_df, recent_tags):
    """ Main function to generate one I&W report per indicator. """
    # Combine vt_df and otx_df on 'search_term'
    combined_df = pd.merge(vt_df, otx_df, on='search_term', how='outer', suffixes=('_vt', '_otx'))

    # Consolidate sources into a single column
    sources_df = consolidate_sources(vt_df, otx_df)
    combined_df = pd.merge(combined_df, sources_df, on='search_term', how='left')

    # Merge recent_tags and drop 'summary' to avoid duplication
    if not recent_tags.empty:
        combined_df = pd.merge(
            combined_df,
            recent_tags[['summary', 'observations', 'type', 'partners', 'observed_date', 'webLink']],
            left_on='search_term',
            right_on='summary',
            how='left'
        )
        combined_df.drop(columns=['summary'], inplace=True)
        display(combined_df)
    # Generate individual reports per unique indicator
    current_date = pd.Timestamp.now().strftime("%Y-%m-%d")
    for indicator in combined_df['search_term'].unique():
        indicator_df = combined_df[combined_df['search_term'] == indicator]
        sanitized_indicator = indicator.replace(":", "_").replace("/", "_")
        output_file = os.path.join(OUTPUT_DIR, f"I&W_Report_{sanitized_indicator}_{current_date}.docx")
        #fill_word_template(TEMPLATE_PATH, output_file, indicator_df)
    
if __name__ == "__main__":
    main(vt_df, otx_df, recent_tags)



KeyError: 'search_term'